<a href="https://colab.research.google.com/github/ZeeMurphy/Data_205_Capstone_Project/blob/main/ingestion/notebook/Cognitive_Decline_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cognitive Decline Risk Factors in U.S. Older Adults
## Data Ingestion and Exploratory Data Analysis (EDA)

**Dataset:** CDC BRFSS — Alzheimer's Disease and Healthy Aging Data (2015–2022)  
**Tools:** Python, Pandas, Matplotlib, Seaborn

The goal of this project is to explore what health and behavioral factors are linked to cognitive decline (memory loss) in older adults across U.S. states. This notebook covers loading and cleaning the data, then exploring it through charts and statistics to answer my research questions.

**Research questions I'm exploring here:**
- RQ1: Which states have the highest cognitive decline rates?
- RQ2–RQ5: How do depression, physical inactivity, poor health, and disability relate to cognitive decline?
- RQ8: Does cognitive decline differ between the 50–64 and 65+ age groups?
- RQ9: Are early-stage memory complaints linked to more severe cognitive decline?

*(RQ6, RQ7, RQ10 require Census income/education data — coming in the next phase)*

## Part 1: Load and Clean the Data

I downloaded the CDC dataset as a CSV file from data.cdc.gov. I chose to use a static file (rather than pulling live from an API) because this dataset doesn't change in real time and using a fixed file makes my analysis reproducible — anyone running this notebook will get the same results.

The dataset has 284,142 rows and 31 columns. Each row represents one survey answer for one question, in one location, in one year, for one age subgroup. This is called **long format** — I'll reshape it later.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Load the dataset
# In Google Colab: upload the CSV first using the folder icon on the left sidebar
df = pd.read_csv(
    "/content/Alzheimers_Disease_and_Healthy_Aging_Data_20260304.csv",
    engine="python",
    on_bad_lines="skip"
)

print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head()

Rows: 284,142  |  Columns: 31


,RowId,YearStart,YearEnd,LocationAbbr,LocationDesc,Datasource,Class,Topic,Question,Data_Value_Unit,...,Stratification2,Geolocation,ClassID,TopicID,QuestionID,LocationID,StratificationCategoryID1,StratificationID1,StratificationCategoryID2,StratificationID2
0,BRFSS~2015~2015~66~Q35~TOC03~AGE~SEX,2015,2015,GU,Guam,BRFSS,Overall Health,Recent activity limitations in past month,Mean number of days with activity limitations ...,Number,...,Female,POINT (144.793731 13.444304),C01,TOC03,Q35,66,AGE,AGE_OVERALL,SEX,FEMALE
1,BRFSS~2015~2015~25~Q27~TMC03~AGE~SEX,2015,2015,MA,Massachusetts,BRFSS,Mental Health,Lifetime diagnosis of depression,Percentage of older adults with a lifetime dia...,%,...,Male,POINT (-72.08269067499964 42.27687047000046),C05,TMC03,Q27,25,AGE,65PLUS,SEX,MALE
2,BRFSS~2015~2015~9002~Q43~TOC11~AGE~SEX,2015,2015,MDW,Midwest,BRFSS,Overall Health,Arthritis among older adults,Percentage of older adults ever told they have...,%,...,Male,NaN,C01,TOC11,Q43,9002,AGE,5064,SEX,MALE
3,BRFSS~2015~2015~27~Q03~TMC01~AGE~SEX,2015,2015,MN,Minnesota,BRFSS,Mental Health,Frequent mental distress,Percentage of older adults who are experiencin...,%,...,Male,POINT (-94.79420050299967 46.35564873600049),C05,TMC01,Q03,27,AGE,AGE_OVERALL,SEX,MALE
4,BRFSS~2015~2015~29~Q43~TOC11~AGE~OVERALL,2015,2015,MO,Missouri,BRFSS,Overall Health,Arthritis among older adults,Percentage of older adults ever told they have...,%,...,NaN,POINT (-92.56630005299968 38.635790776000476),C01,TOC11,Q43,29,AGE,AGE_OVERALL,OVERALL,OVERALL


In [ ]:
# Check data types and see which columns have missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 284142 entries, 0 to 284141
Data columns (total 31 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   RowId                       284142 non-null  object 
 1   YearStart                   284142 non-null  int64  
 2   YearEnd                     284142 non-null  int64  
 3   LocationAbbr                284142 non-null  object 
 4   LocationDesc                284142 non-null  object 
 5   Datasource                  284142 non-null  object 
 6   Class                       284142 non-null  object 
 7   Topic                       284142 non-null  object 
 8   Question                    284142 non-null  object 
 9   Data_Value_Unit             284142 non-null  object 
 10  DataValueTypeID             284142 non-null  object 
 11  Data_Value_Type             284142 non-null  object 
 12  Data_Value                  192808 non-null  float64
 13  Data_Value_Alt

In [ ]:
# --- Data Cleaning ---
# Step 1: Make a copy so I don't change the original
df_clean = df.copy()

# Step 2: Replace different versions of missing values with one standard (pd.NA)
df_clean = df_clean.replace(["", " ", "NA", "N/A"], pd.NA)

# Step 3: Convert Data_Value and YearStart to numbers
# errors='coerce' turns anything that can't be converted into NaN instead of crashing
df_clean["Data_Value"] = pd.to_numeric(df_clean["Data_Value"], errors="coerce")
df_clean["YearStart"]  = pd.to_numeric(df_clean["YearStart"],  errors="coerce")

# Step 4: Drop rows missing the value or the year — they can't be used in any analysis
before = len(df_clean)
df_clean = df_clean.dropna(subset=["Data_Value", "YearStart"]).copy()

print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning:  {len(df_clean):,}")
print(f"Rows dropped:         {before - len(df_clean):,}")

Rows before cleaning: 284,142
Rows after cleaning:  192,808
Rows dropped:         91,334


In [ ]:
# Quick look at what's in the key columns
print("Unique locations:", df_clean['LocationAbbr'].nunique())
print(sorted(df_clean['LocationAbbr'].unique()))
print("\nAge group options:", df_clean['Stratification1'].unique())
print("\nHealth topic classes:", sorted(df_clean['Class'].unique()))

Unique locations: 59
['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'GU', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'MDW', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NRE', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'PR', 'RI', 'SC', 'SD', 'SOU', 'TN', 'TX', 'US', 'UT', 'VA', 'VI', 'VT', 'WA', 'WEST', 'WI', 'WV', 'WY']

Age group options: ['Overall' '65 years or older' '50-64 years']

Health topic classes: ['Caregiving', 'Cognitive Decline', 'Mental Health', 'Nutrition/Physical Activity/Obesity', 'Overall Health', 'Screenings and Vaccines', 'Smoking and Alcohol Use']


The dataset includes 59 locations — not just the 50 states. It also includes U.S. territories like Puerto Rico and Guam, plus regional averages for the Midwest, South, Northeast, and West. I'll filter these out and keep only the 50 states + D.C. for my state-level analysis.

Each question also appears three times per state-year: once for the full 'Overall' group, once for ages 50–64, and once for ages 65+. I'll use 'Overall' for most analysis to avoid counting the same state three times.

In [ ]:
# Check the exact wording of cognitive decline questions — need exact strings for filtering
print("Cognitive Decline questions:")
for q in df_clean[df_clean['Class'] == 'Cognitive Decline']['Question'].unique():
    print(" -", q)

Cognitive Decline questions:
 - Percentage of older adults who reported subjective cognitive decline or memory loss that interferes with their ability to engage in social activities or household chores
 - Percentage of older adults with subjective cognitive decline or memory loss who reported talking with a health care professional about it
 - Percentage of older adults who reported subjective cognitive decline or memory loss that is happening more often or is getting worse in the preceding 12 months
 - Percentage of older adults who reported that as a result of subjective cognitive decline or memory loss that they need assistance with day-to-day activities


## Part 2: Check Data Coverage by Year

Before building my analysis dataset, I want to check how many states reported cognitive decline data each year. The cognitive decline module in BRFSS is optional — states choose whether to include it. This means some years have much less data than others, which is an important limitation to keep in mind.

In [ ]:
cognitive_questions = [
    "Percentage of older adults who reported subjective cognitive decline or memory loss that interferes with their ability to engage in social activities or household chores",
    "Percentage of older adults who reported subjective cognitive decline or memory loss that is happening more often or is getting worse in the preceding 12 months",
    "Percentage of older adults who reported that as a result of subjective cognitive decline or memory loss that they need assistance with day-to-day activities",
    "Percentage of older adults with subjective cognitive decline or memory loss who reported talking with a health care professional about it"
]

# Non-state locations to exclude
exclude = ["US", "NRE", "MDW", "SOU", "WEST"]

coverage = df_clean[
    (df_clean["Question"].isin(cognitive_questions)) &
    (~df_clean["LocationAbbr"].isin(exclude)) &
    (df_clean["Stratification1"] == "Overall")
].groupby(["Question", "YearStart"])["LocationAbbr"].nunique().unstack(fill_value=0)

coverage.index = ["Interferes with activities", "Getting worse", "Needs assistance", "Talked to doctor"]
print("States reporting each cognitive decline question per year:")
print(coverage.to_string())

States reporting each cognitive decline question per year:
YearStart                   2015  2016  2017  2018  2019  2020  2021  2022
Interferes with activities    35    21    10     6    49    23    15    18
Getting worse                 35    21    10     6    49    23    15    18
Needs assistance              35    21    10     6    49    23    15    18
Talked to doctor              35    21    10     6    49    23    15    18


Coverage dropped sharply in 2017–2018 (only 10 and 6 states), then recovered in 2019 (49 states). This means year-to-year comparisons need to be interpreted carefully — the national average in 2018 is based on only 6 states, which may not represent the whole country.

## Part 3: Define Variables and Build Analysis Dataset

I'm selecting one cognitive decline question as my main outcome (target), plus six health and behavioral variables as predictors. Then I'll filter and reshape the data into wide format — one row per state per year, with each variable as its own column.

In [ ]:
# Main outcome variable (target)
target = "Percentage of older adults who reported subjective cognitive decline or memory loss that interferes with their ability to engage in social activities or household chores"

# The other three cognitive decline questions (used in RQ9)
q_worse      = "Percentage of older adults who reported subjective cognitive decline or memory loss that is happening more often or is getting worse in the preceding 12 months"
q_assistance = "Percentage of older adults who reported that as a result of subjective cognitive decline or memory loss that they need assistance with day-to-day activities"
q_talked     = "Percentage of older adults with subjective cognitive decline or memory loss who reported talking with a health care professional about it"

# Six risk factor variables (predictors for RQ2–RQ5)
risk_factors = [
    "Percentage of older adults with a lifetime diagnosis of depression",
    "Percentage of older adults who are experiencing frequent mental distress",
    "Percentage of older adults who have not had any leisure time physical activity in the past month",
    'Percentage of older adults who self-reported that their health is "fair" or "poor"',
    "Physically unhealthy days (mean number of days in past month)",
    "Percentage of older adults who report having a disability (includes limitations related to sensory or mobility impairments or a physical, mental, or emotional condition)"
]

# Short names for chart labels
short_names = {
    target: "Cognitive Decline (%)",
    q_worse: "Getting Worse",
    q_assistance: "Needs Assistance",
    q_talked: "Talked to Doctor",
    'Percentage of older adults who self-reported that their health is "fair" or "poor"': "Poor Health",
    "Percentage of older adults who have not had any leisure time physical activity in the past month": "Physical Inactivity",
    "Percentage of older adults who report having a disability (includes limitations related to sensory or mobility impairments or a physical, mental, or emotional condition)": "Disability",
    "Percentage of older adults who are experiencing frequent mental distress": "Mental Distress",
    "Physically unhealthy days (mean number of days in past month)": "Unhealthy Days",
    "Percentage of older adults with a lifetime diagnosis of depression": "Depression"
}

# U.S. states + DC only
us_states = [
    'AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN',
    'IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV',
    'NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN',
    'TX','UT','VT','VA','WA','WV','WI','WY','DC'
]

print("Variables set up.")

Variables set up.


In [ ]:
# Filter to only the questions I need, Overall age group, and U.S. states only
# Note: filtering to states only is important — without this, Puerto Rico and
# regional averages like 'South' would appear as rows in my charts
all_questions = [target] + risk_factors

df_filtered = df_clean[
    (df_clean["Question"].isin(all_questions)) &
    (df_clean["Stratification1"] == "Overall") &
    (df_clean["LocationAbbr"].isin(us_states))
].copy()

# Reshape from long to wide format
# Each row = one state + one year, each question becomes its own column
# aggfunc='mean' averages across the sex/race breakdowns that exist within 'Overall'
pivot = df_filtered.pivot_table(
    index=["LocationAbbr", "LocationDesc", "YearStart"],
    columns="Question",
    values="Data_Value",
    aggfunc="mean"
).reset_index()
pivot.columns.name = None

print(f"Analysis dataset shape: {pivot.shape}")
print(f"This means {pivot.shape[0]} state-year combinations across {pivot['LocationAbbr'].nunique()} states")
pivot.head(3)

Analysis dataset shape: (406, 10)
This means 406 state-year combinations across 51 states


,LocationAbbr,LocationDesc,YearStart,Percentage of older adults who are experiencing frequent mental distress,Percentage of older adults who have not had any leisure time physical activity in the past month,"Percentage of older adults who report having a disability (includes limitations related to sensory or mobility impairments or a physical, mental, or emotional condition)",Percentage of older adults who reported subjective cognitive decline or memory loss that interferes with their ability to engage in social activities or household chores,"Percentage of older adults who self-reported that their health is ""fair"" or ""poor""",Percentage of older adults with a lifetime diagnosis of depression,Physically unhealthy days (mean number of days in past month)
0,AK,Alaska,2015,9.24,31.04,NaN,NaN,20.98,16.68,4.84
1,AK,Alaska,2016,9.56,24.60,30.24,29.475,22.20,12.98,5.74
2,AK,Alaska,2017,7.10,26.36,28.66,NaN,21.44,15.84,5.18


## Part 4: Missing Value Analysis

Now I'll check how much data is missing in the pivot table. A quick note on why the numbers might look small: the missing percentages are calculated out of **406 state-year rows**, not the original 284,142 rows. After pivoting, each row represents one state in one year, so there are only about 406 possible combinations (51 states × 8 years).

The cognitive decline variable has the most missing data because it comes from an **optional** BRFSS module, states choose whether to run it each year. The other variables (depression, physical inactivity, poor health) come from **core** modules that nearly every state runs every year, which is why they have almost no missing data.

In [ ]:
# Count missing values per column
missing = pd.DataFrame({
    'Missing Count': pivot.isna().sum(),
    'Missing %': (pivot.isna().mean() * 100).round(1)
})
missing.index = [short_names.get(c, c) for c in missing.index]
print("Missing values (out of", len(pivot), "state-year rows):")
missing[missing['Missing Count'] > 0]

Missing values (out of 406 state-year rows):


,Missing Count,Missing %
Disability,51,12.6
Cognitive Decline (%),233,57.4


In [ ]:
# Drop rows where the main cognitive decline variable is missing
# I keep rows where only a risk factor is missing — those can still be used in some charts
before = len(pivot)
pivot_clean = pivot.dropna(subset=[target]).copy()

print(f"Rows before: {before}  |  Rows after: {len(pivot_clean)}  |  Dropped: {before - len(pivot_clean)}")
print(f"States with cognitive decline data: {pivot_clean['LocationDesc'].nunique()}")
print(f"Years covered: {sorted(pivot_clean['YearStart'].unique())}")

Rows before: 406  |  Rows after: 173  |  Dropped: 233
States with cognitive decline data: 51
Years covered: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
